# 01 - Build `ml.bus_matching_candidate_pairs`

First notebook for the Bus Matching model: builds the full universe of
*plausible* `bus_id` <-> `device_id` pairs from the two vehicle
dictionaries, for a later ML model to pick the real match out of. Unlike
`ml.trip_validity_bus_avl_match` (the Trip Validity model's crosswalk,
which picks one arbitrary-but-deterministic match per `bus_id`), this
notebook keeps **every** plausible pair, many-to-many, and tags each with
where it came from -- collapsing to a single best match is the next
model's job, not this notebook's.

`bus_id` uses the exact same column name and normalization as
`ml.trip_validity_final.bus_id`: strip everything but digits, then
left-pad with zeros to 5 characters (only when shorter -- never
truncate, so a noisier/longer code is kept as-is rather than corrupted).

A pair is only kept when **both** sides are independently corroborated:
`bus_id` must be one of the `ml.trip_validity_final.bus_id` values
**and** `device_id` must actually appear in `silver.avl_pings` during the
window below. A pair where either side has no independent evidence isn't
useful for the matching model -- there'd be nothing on that side to
derive features from or compare against.

## The two sources

- `silver.dictionary_device` (2,219 rows, 1 snapshot): `codigo` normalizes
  to `bus_id` as above (6 rows strip to empty and are dropped, 401 have
  no `device_id` at all and are dropped too, since there's nothing to
  pair). Kept only when `bus_id` is one of the 1,730 distinct `bus_id`s
  in `ml.trip_validity_final` **and** `device_id` actually pinged in
  `silver.avl_pings` during the window.
- `silver.dictionary_vehicle` (4,530 rows, 1 snapshot): `cod_veiculo`
  normalizes to `bus_id` the same way (72 rows strip to empty and are
  dropped). `id_veiculo` is always plain-integer text and equals
  `avl_pings.vehicle_id` once cast (confirmed in
  `ml/trip_validity_model/notebooks/04_avl_positions.ipynb`), so every
  `device_id` that vehicle actually pinged from during the window becomes
  a candidate -- kept only for the vehicles whose normalized `bus_id` is
  also one of `ml.trip_validity_final`'s.

## Why "November 2023 +/- one day" is materialized once, up front

`silver.avl_pings` is indexed on `metric_timestamp` for every monthly
partition, but the covering `(vehicle_id, metric_timestamp)` /
`(device_id, metric_timestamp)` indexes only exist on the November 2023
partition (backfilled in notebook 04) -- October and December have
neither. Probing `avl_pings` once per dictionary row (thousands of
probes) against those two unindexed partitions would mean a sequential
scan of each on every probe. Instead, Stage 0 below does one single
`SELECT DISTINCT (vehicle_id, device_id)` scan across the whole window
into a temp table (confirmed live: ~95s for November alone via the
timestamp index), then everything downstream joins against that small
temp table instead of `avl_pings` directly.


In [1]:
import os
from pathlib import Path

import psycopg

In [2]:
_root = Path.cwd()
while not (_root / "pyproject.toml").exists():
    _root = _root.parent
os.chdir(_root)
os.environ.setdefault("RAW_DATA_ROOT", str(_root))

'/home/victor/repos/opa-database'

In [3]:
from opa_database.config import settings

WINDOW_START = "2023-10-31"
WINDOW_END = "2023-12-02"

conn = psycopg.connect(settings.db_dsn)
conn.execute("CREATE SCHEMA IF NOT EXISTS ml;")
conn.commit()
print("ml schema ready")

ml schema ready


## Stage 0 - materialize the window's `(vehicle_id, device_id)` roster

One row per distinct vehicle/device pairing actually seen in
`silver.avl_pings` during `[WINDOW_START, WINDOW_END)`. Indexed on both
columns so the two joins below (device_id membership check, vehicle_id
lookup) are cheap against this small table instead of `avl_pings`
itself.

In [4]:
conn.execute("DROP TABLE IF EXISTS avl_window_vehicle_device;")
conn.execute(
    """
    CREATE TEMP TABLE avl_window_vehicle_device AS
    SELECT DISTINCT vehicle_id, device_id
    FROM silver.avl_pings
    WHERE metric_timestamp >= %(start)s AND metric_timestamp < %(end)s;
    """,
    {"start": WINDOW_START, "end": WINDOW_END},
)
conn.execute("CREATE INDEX ON avl_window_vehicle_device (vehicle_id);")
conn.execute("CREATE INDEX ON avl_window_vehicle_device (device_id);")
conn.execute("ANALYZE avl_window_vehicle_device;")
conn.commit()

with conn.cursor() as cur:
    cur.execute("SELECT count(*) FROM avl_window_vehicle_device;")
    print("distinct (vehicle_id, device_id) pairs in window:", cur.fetchone()[0])

distinct (vehicle_id, device_id) pairs in window: 1498


## Stage 1 - `ml.bus_matching_candidate_pairs`

`bus_id` matches `ml.trip_validity_final.bus_id`'s own column name.
`PRIMARY KEY (bus_id, device_id, origin)`, deliberately not just
`(bus_id, device_id)`: a pair corroborated by both dictionaries should
show up as two rows, not collapse into one -- being backed by both
sources is itself a signal worth keeping for the model that eventually
picks the real match.

In [5]:
conn.execute("""
    DROP TABLE IF EXISTS ml.bus_matching_candidate_pairs CASCADE;

    CREATE TABLE ml.bus_matching_candidate_pairs (
        bus_id      text NOT NULL,
        device_id   text NOT NULL,
        origin      text NOT NULL
            CHECK (origin IN ('dictionary_device', 'dictionary_vehicle')),
        PRIMARY KEY (bus_id, device_id, origin)
    );
""")
conn.commit()
print("ml.bus_matching_candidate_pairs created")

ml.bus_matching_candidate_pairs created


## Stage 2 - pairs from `silver.dictionary_device`

`codigo` -> digits-only -> zero-padded to 5 -> `bus_id`. Rows with no
`device_id` or that strip to an empty string are dropped (nothing to
pair). Kept only when `bus_id` is a known `ml.trip_validity_final.bus_id`
**and** `device_id` is in the window's AVL roster.

In [6]:
conn.execute("""
    WITH normalized AS (
        SELECT
            regexp_replace(codigo, '[^0-9]', '', 'g') AS digits,
            device_id
        FROM silver.dictionary_device
        WHERE device_id IS NOT NULL
    ),
    padded AS (
        SELECT
            CASE WHEN length(digits) < 5 THEN lpad(digits, 5, '0') ELSE digits END
                AS bus_id,
            device_id
        FROM normalized
        WHERE digits <> ''
    )
    INSERT INTO ml.bus_matching_candidate_pairs (bus_id, device_id, origin)
    SELECT DISTINCT p.bus_id, p.device_id, 'dictionary_device'
    FROM padded p
    WHERE p.bus_id IN (SELECT DISTINCT bus_id FROM ml.trip_validity_final)
      AND p.device_id IN (SELECT device_id FROM avl_window_vehicle_device);
""")
conn.commit()

with conn.cursor() as cur:
    cur.execute(
        """
        SELECT count(*) FROM ml.bus_matching_candidate_pairs
        WHERE origin = 'dictionary_device';
        """
    )
    print("dictionary_device pairs kept:", cur.fetchone()[0])

dictionary_device pairs kept: 1160


## Stage 3 - pairs from `silver.dictionary_vehicle`

`cod_veiculo` normalizes to `bus_id` the same way as Stage 2.
`id_veiculo::integer` joins the Stage 0 roster on `vehicle_id` to pull
every `device_id` that vehicle actually pinged from in-window -- the
join itself guarantees the `device_id` side, so the only extra filter
needed here is the same `bus_id` membership check as Stage 2.

In [7]:
conn.execute("""
    WITH normalized AS (
        SELECT
            regexp_replace(cod_veiculo, '[^0-9]', '', 'g') AS digits,
            id_veiculo::integer AS vehicle_id
        FROM silver.dictionary_vehicle
    ),
    padded AS (
        SELECT
            CASE WHEN length(digits) < 5 THEN lpad(digits, 5, '0') ELSE digits END
                AS bus_id,
            vehicle_id
        FROM normalized
        WHERE digits <> ''
    )
    INSERT INTO ml.bus_matching_candidate_pairs (bus_id, device_id, origin)
    SELECT DISTINCT p.bus_id, w.device_id, 'dictionary_vehicle'
    FROM padded p
    JOIN avl_window_vehicle_device w ON w.vehicle_id = p.vehicle_id
    WHERE p.bus_id IN (SELECT DISTINCT bus_id FROM ml.trip_validity_final);
""")
conn.commit()

with conn.cursor() as cur:
    cur.execute(
        """
        SELECT count(*) FROM ml.bus_matching_candidate_pairs
        WHERE origin = 'dictionary_vehicle';
        """
    )
    print("dictionary_vehicle pairs kept:", cur.fetchone()[0])

dictionary_vehicle pairs kept: 1122


## Stage 4 - index and summarize

In [8]:
conn.execute("""
    CREATE INDEX bus_matching_candidate_pairs_bus_id_idx
        ON ml.bus_matching_candidate_pairs (bus_id);
    CREATE INDEX bus_matching_candidate_pairs_device_id_idx
        ON ml.bus_matching_candidate_pairs (device_id);
    ANALYZE ml.bus_matching_candidate_pairs;
""")
conn.commit()

with conn.cursor() as cur:
    cur.execute("""
        SELECT
            origin,
            count(*) AS pairs,
            count(DISTINCT bus_id) AS distinct_bus_ids,
            count(DISTINCT device_id) AS distinct_device_ids
        FROM ml.bus_matching_candidate_pairs
        GROUP BY origin
        ORDER BY origin;
    """)
    for row in cur.fetchall():
        print(row)

    cur.execute("SELECT count(*) FROM ml.bus_matching_candidate_pairs;")
    print("total rows:", cur.fetchone()[0])

    cur.execute("""
        SELECT count(*) FROM (
            SELECT bus_id, device_id
            FROM ml.bus_matching_candidate_pairs
            GROUP BY bus_id, device_id
            HAVING count(DISTINCT origin) = 2
        ) both_sources;
    """)
    print("pairs corroborated by both dictionaries:", cur.fetchone()[0])

('dictionary_device', 1160, 1160, 1160)
('dictionary_vehicle', 1122, 1113, 1122)
total rows: 2282
pairs corroborated by both dictionaries: 610
